# 🤝 ISOM 260: Multi-Agent Systems in Practice

**Session 8 — Multi-Agent Systems** | Suffolk University | Prof. Hasan Arslan

---

## From Session 7 to Session 8: Your Agent Learns to Collaborate

In Session 7, your agent learned to **read CloudSync Solutions' documents** using RAG. It searches the company knowledge base before answering, grounding every response in real data.

But Sarah from Marketing still has to wait while **ONE agent** tries to research competitors, analyze trends, AND write her report. What if she had a team?

**Today, your agent learns to collaborate.** You'll build a team of specialized agents — each with a focused role — that pass work to each other like departments in a company.

```
AI Agent = LLM + Tools + ReAct Reasoning + RAG + 🤝 Agent Teams (NEW!)
```

One agent is useful. A team of agents is powerful. Build pipelines, coordinate handoffs, and manage agent teams.

---

## 🚀 Part 1: Setup (2 minutes)

Same setup as Sessions 4, 5, 6, and 7 — install the SDK and connect your API key.

In [ ]:
# Install dependencies
# anthropic = Claude SDK (same as Sessions 4-7)
!pip install -q anthropic

In [ ]:
# Set your API key
# Same key you've been using since Session 4 — it still works!
import os
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon in the left sidebar → Add secret named ANTHROPIC_API_KEY
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ API key loaded from Colab Secrets!")
except Exception:
    # Option 2: Enter manually
    os.environ["ANTHROPIC_API_KEY"] = input("Enter your Anthropic API key: ")
    print("✅ API key set manually!")

In [ ]:
# Verify connection
from anthropic import Anthropic
import time

client = Anthropic()
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say 'Multi-agent systems ready!' in exactly those words."}]
)
print(response.content[0].text)
print(f"✅ Connected! Model: claude-sonnet-4-20250514")

---

## 🛠️ Part 2: Helper Utilities

Before building our agents, we need a helper function. `call_agent()` wraps the Claude API call and adds **timing stats** and **token tracking** — so we can see exactly how long each agent takes and how much it costs.

This is the same pattern you’d use in production: every agent call gets logged.

In [ ]:
def call_agent(system_prompt, user_message, agent_name="Agent"):
    """Call an agent and return response with timing stats."""
    start = time.time()
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=2048,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    elapsed = time.time() - start
    result = response.content[0].text
    tokens_in = response.usage.input_tokens
    tokens_out = response.usage.output_tokens
    print(f"🤖 {agent_name} responded in {elapsed:.1f}s | Tokens: {tokens_in} in → {tokens_out} out")
    return result

print("✅ call_agent() helper ready!")

---

## 🎯 Part 3: The Single-Agent Baseline

Before building a team, let’s see what **ONE agent** can do when asked to research, analyze, AND write — all by itself.

This is our **baseline**. We’ll compare it to the multi-agent pipeline later.

| Watch For | Why It Matters |
|-----------|---------------|
| Output quality | Does one agent do everything well? |
| Depth of analysis | Can it research AND analyze deeply? |
| Time taken | How long for one agent to do it all? |

In [ ]:
SINGLE_AGENT_PROMPT = """You are a business research assistant. When given a topic:
1. Research key facts and statistics about the topic
2. Analyze patterns and trends
3. Write a clear executive summary with recommendations

Be thorough but concise. Include specific data points where possible."""

query = "CloudSync Solutions wants to expand into the healthcare market. Research the competitive landscape, regulatory requirements, and go-to-market strategy for selling cloud document management to hospitals and clinics."

print("📝 Running single agent on full task...")
print("=" * 60)
single_result = call_agent(SINGLE_AGENT_PROMPT, query, "Single Agent")
print("\n" + single_result)

---

## 🔍 Part 4: Build the Researcher Agent

Now we start building our **specialized team**. First up: the Researcher.

Like a company’s research department — they don’t write the final report, they **gather the facts**. Their job is to find information and structure it clearly for the next agent.

**Specialization is the key insight.** Instead of one generalist, we give each agent a focused role with a structured output format.

🔵 **Researcher = Cyan in our diagrams** — the agent that gathers raw facts and data.

In [ ]:
RESEARCHER_PROMPT = """You are a specialized Research Agent. Your ONLY job is to gather facts.

When given a research topic, return EXACTLY this format:

## Key Facts
- [Fact 1 with source/year if available]
- [Fact 2]
- [Fact 3]
(at least 5 facts)

## Statistics
- [Stat 1 with number and context]
- [Stat 2]
(at least 3 statistics)

## Trends
- [Trend 1 with direction: growing/declining/stable]
- [Trend 2]
(at least 3 trends)

## Confidence Level
[HIGH/MEDIUM/LOW] - Brief explanation of confidence

IMPORTANT: Only report facts you are confident about. If unsure, say so.
Do NOT analyze, recommend, or write summaries. Just gather facts."""

query = "CloudSync Solutions wants to expand into the healthcare market. Research the competitive landscape, regulatory requirements, and go-to-market strategy for selling cloud document management to hospitals and clinics."

print("🔍 Researcher Agent working...")
print("=" * 60)
researcher_output = call_agent(RESEARCHER_PROMPT, query, "🔍 Researcher")
print("\n" + researcher_output)

---

## 📊 Part 5: Build the Analyst Agent

The Analyst receives the Researcher’s structured output and **finds patterns**. This is where raw data becomes insight.

Notice the handoff: the Researcher’s output becomes the Analyst’s input. This is **message passing** — the simplest coordination pattern.

🟣 **Analyst = Magenta in our diagrams** — the agent that identifies patterns and generates recommendations.

| Watch For | Why It Matters |
|-----------|---------------|
| Does it ADD value? | Or just rephrase what Researcher said? |
| Pattern quality | Are the patterns insightful or obvious? |
| Recommendation specificity | Vague advice vs actionable steps |

In [ ]:
ANALYST_PROMPT = """You are a specialized Analyst Agent. You receive research findings and identify patterns.

Your input will be structured research with Key Facts, Statistics, and Trends.

Return EXACTLY this format:

## Patterns Identified
- [Pattern 1: What the data reveals]
- [Pattern 2]
- [Pattern 3]

## Insights
- [Insight 1: Non-obvious conclusion from the data]
- [Insight 2]

## Recommendations
1. [Specific, actionable recommendation]
2. [Another recommendation]
3. [Third recommendation]

## Confidence Assessment
[Your confidence in these patterns: HIGH/MEDIUM/LOW]
[Which findings were strongest vs weakest]

IMPORTANT: Build on the research — don't just restate it. Find what the researcher missed.
Do NOT add new facts. Only analyze what was provided."""

analyst_input = f"""Analyze these research findings and identify patterns:

{researcher_output}"""

print("📊 Analyst Agent working...")
print("=" * 60)
analyst_output = call_agent(ANALYST_PROMPT, analyst_input, "📊 Analyst")
print("\n" + analyst_output)

---

## ✏️ Part 6: Build the Editor Agent

The Editor is **quality control**. It receives both the original research AND the analysis, then:
- **Fact-checks** every major claim
- **Improves clarity** of the final output
- **Scores quality** across multiple dimensions

This is the agent that catches errors the others miss. In a real system, this is where you’d add human review gates.

🟢 **Editor = Green in our diagrams** — the agent that verifies, polishes, and scores.

In [ ]:
EDITOR_PROMPT = """You are a specialized Editor Agent. You receive both original research AND analysis.

Your job: fact-check, improve clarity, and score quality.

Return EXACTLY this format:

## Fact Check
- ✅ [Verified fact] — Consistent between research and analysis
- ✅ [Another verified fact]
- ⚠️ [Questionable claim] — Why it's questionable
- ❌ [Incorrect/unsupported claim] — What's wrong
(Check every major claim)

## Improved Executive Summary
[Write a clear, 150-word executive summary combining the best of research + analysis.
Every claim must trace back to the researcher's findings.]

## Quality Score
**Overall: [X]/10**
- Research thoroughness: [X]/10
- Analysis depth: [X]/10
- Factual accuracy: [X]/10
- Actionability: [X]/10

## Suggestions for Improvement
- [What would make this better]

IMPORTANT: Be a tough but fair editor. Flag anything that seems made up or unsupported."""

editor_input = f"""Review this research pipeline output for accuracy and quality.

ORIGINAL RESEARCH:
{researcher_output}

ANALYSIS:
{analyst_output}"""

print("✏️ Editor Agent working...")
print("=" * 60)
editor_output = call_agent(EDITOR_PROMPT, editor_input, "✏️ Editor")
print("\n" + editor_output)

---

## 🔗 Part 7: Wire the Full Pipeline

Now let’s wire all three agents together into **one function**. This is the complete pipeline:

```
🔍 Researcher → 📊 Analyst → ✏️ Editor
```

Each agent receives the previous agent’s output. The Editor gets BOTH the research and analysis — so it can fact-check the full chain.

This is the **pipeline pattern** — the most common multi-agent architecture. Master this, and you can adapt it to any domain.

In [ ]:
def run_pipeline(query):
    """Run the full Researcher → Analyst → Editor pipeline."""
    print(f"🚀 Pipeline started for: '{query}'")
    print("=" * 60)
    pipeline_start = time.time()

    # Stage 1: Research
    print("\n📍 Stage 1/3: Researcher")
    researcher_output = call_agent(RESEARCHER_PROMPT, query, "🔍 Researcher")

    # Stage 2: Analysis
    print("\n📍 Stage 2/3: Analyst")
    analyst_input = f"Analyze these research findings and identify patterns:\n\n{researcher_output}"
    analyst_output = call_agent(ANALYST_PROMPT, analyst_input, "📊 Analyst")

    # Stage 3: Editing
    print("\n📍 Stage 3/3: Editor")
    editor_input = f"Review this research pipeline output for accuracy and quality.\n\nORIGINAL RESEARCH:\n{researcher_output}\n\nANALYSIS:\n{analyst_output}"
    editor_output = call_agent(EDITOR_PROMPT, editor_input, "✏️ Editor")

    total_time = time.time() - pipeline_start
    print("\n" + "=" * 60)
    print(f"✅ Pipeline complete in {total_time:.1f}s")

    return {
        "query": query,
        "researcher": researcher_output,
        "analyst": analyst_output,
        "editor": editor_output,
        "total_time": total_time
    }

# Test the pipeline with the same CloudSync healthcare query
result = run_pipeline("CloudSync Solutions wants to expand into the healthcare market. Research the competitive landscape, regulatory requirements, and go-to-market strategy for selling cloud document management to hospitals and clinics.")

In [ ]:
print("📋 FINAL EDITED OUTPUT")
print("=" * 60)
print(result["editor"])

---

## ⚖️ Part 8: The CEO Test — Which Output Would You Trust?

CloudSync's CEO needs to make a decision about entering healthcare. She has 5 minutes to read a briefing. Which output — single agent or pipeline — would you hand her?

This is the real test of multi-agent systems: **not whether the code is fancier, but whether the output is better enough to justify the extra complexity.**

In [ ]:
print("🏢 SCENARIO: CloudSync expanding into healthcare")
print("=" * 60)
print()
print("👤 WHAT ONE GENERALIST PRODUCED:")
print("─" * 60)
print(single_result[:1500] if len(single_result) > 1500 else single_result)
print()
print()
print("🤝 WHAT THE SPECIALIST TEAM PRODUCED:")
print("─" * 60)
print(result["editor"][:1500] if len(result["editor"]) > 1500 else result["editor"])
print()
print("💡 DISCUSSION QUESTIONS:")
print("   • Which output would you actually hand to CloudSync's CEO?")
print("   • What did the pipeline catch that the single agent missed?")
print("   • Was the extra time/cost worth it for this decision?")

---

## 📞 Part 9: The Telephone Game — Information Distortion

Here's where things get scary.

**Does information degrade as it passes between agents?** Think of the children's telephone game — a message changes as it passes from person to person. The same thing can happen with AI agents.

Imagine CloudSync's legal team relying on this pipeline for compliance analysis. If the facts shift each time you run it, that's not just a bug — it's a lawsuit.

Let's test it: run the **SAME query** through the pipeline **3 times**. Do you get the same answer?

| Watch For | Why It Matters |
|-----------|---------------|
| Facts that appear in ALL 3 runs | These are reliable — the pipeline consistently surfaces them |
| Facts that appear in ONLY 1 run | These are unstable — possibly hallucinated or cherry-picked |
| Quality score variance | How consistent is the pipeline's self-assessment? |

In [ ]:
print("📞 Running pipeline 3 times on the SAME query...")
print("This tests consistency — a reliable system should give similar answers.\n")

test_query = "What are the top 3 risks CloudSync Solutions faces when entering the healthcare market, and what regulatory compliance is required?"
runs = []

for i in range(3):
    print("\n" + "🔄" * 20)
    print(f"RUN {i+1} OF 3")
    print("🔄" * 20)
    run_result = run_pipeline(test_query)
    runs.append(run_result)

print("\n✅ All 3 runs complete!")
times_str = ", ".join(f"{r['total_time']:.1f}s" for r in runs)
print(f"Times: {times_str}")

---

## 🔍 Part 10: Distortion Analysis

Now let’s compare the 3 runs side by side. This is where multi-agent systems reveal their **reliability** (or lack thereof).

In [ ]:
print("📊 COMPARING 3 PIPELINE RUNS")
print("=" * 60)

for i, run in enumerate(runs):
    separator = "─" * 40
    print(f"\n{separator}")
    print(f"RUN {i+1} — Editor's Summary (first 500 chars):")
    print(separator)
    # Extract just the executive summary portion if possible
    editor_text = run["editor"]
    print(editor_text[:500] + "...")

print("\n" + "=" * 60)
print("🤔 REFLECTION QUESTIONS:")
print("1. How similar are the 3 outputs? (1=identical, 5=completely different)")
print("2. Which specific facts appeared in ALL 3 runs?")
print("3. Which facts appeared in only ONE run? (potential hallucinations)")
print("4. Did the quality scores vary? By how much?")
print("5. If this were a business-critical system, would you trust it?")

---

## 🛡️ Part 11: Add Safeguards — Citation Chains & Structured Handoffs

The telephone game is real. But we can **reduce distortion** with three techniques:

1. **Citation chains** — Every claim gets a unique ID. Downstream agents must cite which facts they’re building on.
2. **Confidence propagation** — Each agent rates its confidence. Low-confidence claims get flagged, not amplified.
3. **Structured JSON handoffs** — Instead of free-text, agents pass structured data. Harder to subtly distort.

Let’s rebuild our pipeline with these safeguards.

In [ ]:
RESEARCHER_V2_PROMPT = """You are a specialized Research Agent v2. Your ONLY job is to gather facts.

Return your findings as structured JSON:
{
  "facts": [
    {"id": "F1", "claim": "...", "source": "...", "confidence": "HIGH/MEDIUM/LOW"},
    {"id": "F2", "claim": "...", "source": "...", "confidence": "HIGH/MEDIUM/LOW"}
  ],
  "statistics": [
    {"id": "S1", "value": "...", "context": "...", "confidence": "HIGH/MEDIUM/LOW"}
  ],
  "trends": [
    {"id": "T1", "direction": "growing/declining/stable", "description": "...", "confidence": "HIGH/MEDIUM/LOW"}
  ]
}

IMPORTANT: Use JSON format exactly. Every claim gets a unique ID and confidence level."""

ANALYST_V2_PROMPT = """You are a specialized Analyst Agent v2. You receive structured research JSON.

Return your analysis as structured JSON:
{
  "patterns": [
    {"id": "P1", "description": "...", "based_on": ["F1", "F2", "S1"], "confidence": "HIGH/MEDIUM/LOW"}
  ],
  "recommendations": [
    {"id": "R1", "action": "...", "rationale": "...", "based_on": ["P1", "F3"], "confidence": "HIGH/MEDIUM/LOW"}
  ]
}

CRITICAL: Every pattern and recommendation must cite fact IDs from the research.
If you can't trace a conclusion back to specific facts, don't include it."""

EDITOR_V2_PROMPT = """You are a specialized Editor Agent v2. You receive research JSON and analysis JSON.

Verify the citation chain: does every analysis claim trace back to a research fact?

Return:
{
  "verified_claims": [
    {"claim_id": "P1", "status": "VERIFIED", "traced_to": ["F1", "F2"]}
  ],
  "broken_chains": [
    {"claim_id": "R2", "status": "UNVERIFIED", "reason": "Cites F7 which doesn't exist in research"}
  ],
  "quality_score": 8.5,
  "executive_summary": "..."
}"""

print("🛡️ Safeguarded pipeline with citation chains...")
print("=" * 60)

query = "Impact of AI agents on customer service operations"

# Run v2 pipeline
r2 = call_agent(RESEARCHER_V2_PROMPT, query, "🔍 Researcher v2")
print("\nResearcher v2 output (first 500 chars):")
print(r2[:500])

a2 = call_agent(ANALYST_V2_PROMPT, f"Analyze this research:\n{r2}", "📊 Analyst v2")
print("\nAnalyst v2 output (first 500 chars):")
print(a2[:500])

e2 = call_agent(EDITOR_V2_PROMPT, f"Verify this pipeline:\n\nRESEARCH:\n{r2}\n\nANALYSIS:\n{a2}", "✏️ Editor v2")
print("\nEditor v2 output:")
print(e2)

---

## 🏆 Mission: CloudSync Needs a 4th Team Member

The CEO loved your pipeline but has feedback: "The research is great, the analysis is solid, but I need someone who knows healthcare."

**Your mission:** Add a 4th agent to the pipeline. Choose your specialist:

- **Healthcare Compliance Officer** — Knows HIPAA inside and out. Reviews every recommendation for regulatory risk before it reaches the CEO's desk.
- **Competitive Intelligence Analyst** — Maps the healthcare cloud market. Who are the incumbents? Where are the gaps CloudSync can exploit?
- **Go-to-Market Strategist** — Turns analysis into an actionable launch plan. Pricing, partnerships, pilot programs, timeline.

Pick one and modify the pipeline below!

In [ ]:
# 🏆 YOUR CODE HERE — Add a 4th agent!
# Uncomment and modify one of these:

# Option A: Fact-Checker
# FACT_CHECKER_PROMPT = """You are a Fact-Checker Agent.
# You receive research findings and verify each claim.
# For each fact, rate it as: VERIFIED, PLAUSIBLE, or SUSPICIOUS.
# Explain your reasoning for each rating."""

# Option B: Summarizer
# SUMMARIZER_PROMPT = """You are a Summarizer Agent.
# You receive a full research report and condense it into
# exactly ONE paragraph (5-7 sentences) for an executive audience.
# Every sentence must be supported by the research."""

# Option C: Domain Expert
# DOMAIN_EXPERT_PROMPT = """You are a Domain Expert Agent for [YOUR INDUSTRY].
# You receive research findings and add industry-specific context:
# - Relevant regulations or compliance requirements
# - Competitive landscape considerations
# - Implementation challenges specific to this industry"""

# Then modify run_pipeline() to include your 4th agent!
print("👆 Uncomment and modify one of the options above!")
print("Then modify run_pipeline() to include your 4th agent.")

---

## 🎯 Part 12: Reflection — The Stack So Far

### What Just Happened

You walked into class knowing how to build a single AI agent. In 2.5 hours, you built a **team of AI specialists** that collaborate, fact-check each other, and produce work that's measurably better than any one agent alone.

That's not a coding exercise. That's the future of knowledge work.

---

Look how far you've come:

```
Session 4: Tool Use           → Your agent can DO things
Session 5: Real APIs           → Your agent talks to the real world
Session 6: ReAct + Design      → Your agent THINKS before acting
Session 7: RAG                 → Your agent KNOWS your company
Session 8: Multi-Agent         → Your agents work as a TEAM ← YOU ARE HERE
Session 9: YOUR Industry       → Apply everything to YOUR career
```

### Key Takeaways

- **Agent teams mirror org charts.** Division of labor, specialization, and coordination costs apply to AI agents just like human teams. Design your agent system like you'd design a department.
- **Watch for the telephone game.** Information degrades across agents. Citation chains and confidence propagation are your safeguards against compounding errors.
- **Multi-agent isn't always better.** Each agent adds latency, cost, and complexity. Use the decision framework: multi-agent for complex, multi-expertise tasks. Single-agent for everything else.
- **The pipeline pattern is your workhorse.** Research → Analysis → Review is the most common multi-agent pattern. Master it and you can adapt it to any domain.

### Homework

1. **Extend:** Add a 4th agent to your pipeline (e.g., a Fact-Checker, Summarizer, or Domain Expert). Test the full pipeline with 3 business questions.
2. **Analyze:** Run your pipeline 5 times on the same question. Document how outputs differ, where distortion occurs, and what safeguards would help.
3. **Design:** Propose a multi-agent system for your target industry (the one you'll use for your final project). Include: agent roles, handoff logic, human gates, and cost estimate.
4. **Read:** Read about Claude Code's multi-agent 'team' feature. Write a paragraph on how it applies the orchestrator pattern from Session 6.

### Next Session

**Session 9: Agents for Your Industry** — Pick your department. Build your agent. Marketing, finance, HR, or operations — apply everything you've learned to YOUR career.